# ASX Instrument Data Pull — Interactive Brokers

Download historical bar data for ASX (Australian Securities Exchange) stocks from Interactive Brokers TWS.

**Prerequisites:**
- TWS (Trader Workstation) or IB Gateway running locally
- API connections enabled in TWS (Edit → Global Configuration → API → Settings)
- Port 7496 (live) or 7497 (paper) open

**This notebook pulls two timeframes:**
- **1-minute bars** — 6 months lookback (IB's practical limit for minute data)
- **End-of-day (1-day) bars** — 10 years lookback

**Example instruments:** CBA, BHP, CSL, WES, NAB (top ASX stocks by market cap)


## 0. Check TWS Connection


In [ ]:
import socket

TWS_HOST = "127.0.0.1"
TWS_PORT = 7496  # 7496 = live, 7497 = paper

try:
    with socket.create_connection((TWS_HOST, TWS_PORT), timeout=3):
        print(f"OK — TWS is accepting connections at {TWS_HOST}:{TWS_PORT}")
except (ConnectionRefusedError, OSError) as e:
    print(f"FAILED — Cannot connect to TWS at {TWS_HOST}:{TWS_PORT}")
    print(f"  Error: {e}")
    print(f"  Make sure TWS/IB Gateway is running and API connections are enabled.")


## 1. Imports and Configuration


In [ ]:
import asyncio
import datetime
import os
from pathlib import Path

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from nautilus_trader.adapters.interactive_brokers.historical.client import (
    HistoricInteractiveBrokersClient,
)
from nautilus_trader.persistence.catalog import ParquetDataCatalog

# === TWS Settings ===
TWS_HOST = "127.0.0.1"
TWS_PORT = 7496
TWS_CLIENT_ID = 2  # Use a different client ID from other notebooks

# === Catalog ===
CATALOG_PATH = Path(
    os.environ.get("NAUTILUS_STORE_PATH", str(Path.cwd().parents[2] / "backtest_catalog"))
)
print(f"Catalog: {CATALOG_PATH}")


## 2. Define ASX Instruments

Each instrument is an `IBContract` with:
- `secType="STK"` — equity/stock
- `exchange="ASX"` — Australian Securities Exchange
- `currency="AUD"` — Australian Dollar
- Price type: `LAST` (last traded price, appropriate for stocks)


In [ ]:
# === ASX Instruments ===
# Add or remove instruments as needed.

ASX_INSTRUMENTS = {
    "CBA": IBContract(secType="STK", symbol="CBA", exchange="ASX", currency="AUD"),
    "BHP": IBContract(secType="STK", symbol="BHP", exchange="ASX", currency="AUD"),
    "CSL": IBContract(secType="STK", symbol="CSL", exchange="ASX", currency="AUD"),
    "WES": IBContract(secType="STK", symbol="WES", exchange="ASX", currency="AUD"),
    "NAB": IBContract(secType="STK", symbol="NAB", exchange="ASX", currency="AUD"),
}

# === Timeframe Configuration ===
# 1-minute: 6 months lookback
# 1-day: 10 years lookback

PULLS = {
    "1-MINUTE-LAST": {
        "lookback": datetime.timedelta(days=180),  # ~6 months
        "use_rth": True,  # Regular trading hours only (avoids sparse after-hours data)
    },
    "1-DAY-LAST": {
        "lookback": datetime.timedelta(days=3650),  # ~10 years
        "use_rth": True,
    },
}

print(f"Instruments: {list(ASX_INSTRUMENTS.keys())}")
print(f"Timeframes: {list(PULLS.keys())}")


## 3. Connect to TWS


In [ ]:
import subprocess
import re

def _kill_stale_ib_connections(current_pid: int, tws_port: int = TWS_PORT):
    """Kill orphaned Python processes holding IB connections."""
    try:
        result = subprocess.run(
            ["lsof", "-i", f":{tws_port}", "-P"],
            capture_output=True, text=True, timeout=5,
        )
        for line in result.stdout.strip().split("\n")[1:]:
            parts = line.split()
            if len(parts) >= 2 and parts[0].lower() == "python":
                pid = int(parts[1])
                if pid != current_pid:
                    print(f"  Killing stale process PID={pid}")
                    subprocess.run(["kill", str(pid)], timeout=5)
    except Exception:
        pass

# Kill stale connections
_kill_stale_ib_connections(os.getpid())

# Connect
client = HistoricInteractiveBrokersClient(
    host=TWS_HOST,
    port=TWS_PORT,
    client_id=TWS_CLIENT_ID,
)

await client.connect()
await asyncio.sleep(2)  # Wait for IB readiness

print(f"Connected to TWS at {TWS_HOST}:{TWS_PORT}")
print(f"Client ID: {TWS_CLIENT_ID}")


## 4. Resolve Instruments

Request full instrument details from IB. This resolves tick sizes, lot sizes, trading hours, etc.


In [ ]:
resolved_instruments = {}

for name, contract in ASX_INSTRUMENTS.items():
    instruments = await client.request_instruments(contracts=[contract])
    if instruments:
        resolved_instruments[name] = instruments[0]
        print(f"  {name}: {instruments[0].id} — resolved OK")
    else:
        print(f"  {name}: FAILED to resolve — check contract details")

print(f"\nResolved {len(resolved_instruments)}/{len(ASX_INSTRUMENTS)} instruments")


## 5. Pull Historical Data

This is the main data pull. For each instrument and timeframe:
1. Chunks the date range into IB-compatible sizes
2. Rate-limits requests (max 55 per 10 minutes)
3. Retries on failure
4. Validates returned data

**Estimated time:**
- 1-minute bars (6 months): ~60 requests per instrument × 3s delay = ~3 minutes each
- 1-day bars (10 years): ~10 requests per instrument × 3s delay = ~30 seconds each
- Total for 5 instruments: ~20 minutes


In [ ]:
import time
from collections import deque

# === Rate Limiter ===
_MAX_REQUESTS = 55
_WINDOW = 600  # seconds
_MIN_DELAY = 3.0  # seconds between requests
_timestamps: deque[float] = deque()


async def _rate_limit():
    """Wait until a request slot is available."""
    now = time.monotonic()
    while _timestamps and (now - _timestamps[0]) > _WINDOW:
        _timestamps.popleft()

    if len(_timestamps) >= _MAX_REQUESTS:
        wait = _WINDOW - (now - _timestamps[0]) + 0.5
        print(f"  [rate] Pacing: {len(_timestamps)}/{_MAX_REQUESTS} requests. Waiting {wait:.0f}s...")
        await asyncio.sleep(wait)

    if _timestamps:
        elapsed = time.monotonic() - _timestamps[-1]
        if elapsed < _MIN_DELAY:
            await asyncio.sleep(_MIN_DELAY - elapsed)

    _timestamps.append(time.monotonic())


# === Chunk Sizes ===
_MAX_CHUNK = {
    60:    (datetime.timedelta(days=1),    "1 D"),    # 1-MINUTE: 1 day per request
    300:   (datetime.timedelta(weeks=1),   "1 W"),    # 5-MINUTE: 1 week per request
    3600:  (datetime.timedelta(days=30),   "1 M"),    # 1-HOUR: 1 month per request
    86400: (datetime.timedelta(days=365),  "1 Y"),    # 1-DAY: 1 year per request
}

_BAR_SECONDS = {
    "1-MINUTE": 60, "5-MINUTE": 300, "1-HOUR": 3600, "1-DAY": 86400,
}


async def pull_bars(
    client,
    contract: IBContract,
    bar_spec: str,
    start: datetime.datetime,
    end: datetime.datetime,
    use_rth: bool = True,
    timeout: int = 10,
    max_retries: int = 3,
) -> list:
    """Pull historical bars with chunking, rate limiting, and retries."""
    size_key = bar_spec.rsplit("-", 1)[0]  # "1-MINUTE-LAST" -> "1-MINUTE"
    bar_seconds = _BAR_SECONDS[size_key]
    chunk_td, duration_str = _MAX_CHUNK[bar_seconds]

    all_bars = []
    current_end = end

    while current_end > start:
        chunk_start = max(current_end - chunk_td, start)

        for attempt in range(max_retries):
            try:
                await _rate_limit()
                bars = await asyncio.wait_for(
                    client.request_bars(
                        bar_specifications=[bar_spec],
                        end_date_time=current_end,
                        duration=duration_str,
                        tz_name="UTC",
                        contracts=[contract],
                        use_rth=use_rth,
                        timeout=timeout,
                    ),
                    timeout=timeout + 5,
                )
                if bars:
                    all_bars.extend(bars)
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"    Retry {attempt + 1}/{max_retries}: {e}")
                    await asyncio.sleep(5)
                else:
                    print(f"    FAILED after {max_retries} attempts: {e}")

        current_end = chunk_start

    # Sort and deduplicate
    all_bars.sort(key=lambda b: b.ts_init)
    seen = set()
    unique = []
    for b in all_bars:
        if b.ts_init not in seen:
            seen.add(b.ts_init)
            unique.append(b)

    return unique


print("pull_bars() ready")


## 6. Execute Pulls

Pull all instruments for all timeframes. Results are stored in memory before writing to the catalog.


In [ ]:
end = datetime.datetime.now(datetime.timezone.utc)
results = {}  # {(instrument_name, bar_spec): bars}

for name, contract in ASX_INSTRUMENTS.items():
    if name not in resolved_instruments:
        print(f"Skipping {name} — not resolved")
        continue

    for bar_spec, config in PULLS.items():
        start = end - config["lookback"]
        key = (name, bar_spec)

        print(f"\n{'='*60}")
        print(f"Pulling {name} {bar_spec}")
        print(f"  Range: {start.date()} → {end.date()}")

        bars = await pull_bars(
            client, contract, bar_spec,
            start=start, end=end,
            use_rth=config["use_rth"],
        )

        results[key] = bars
        print(f"  Got {len(bars)} bars")

print(f"\n{'='*60}")
print(f"Pull complete. {len(results)} datasets ready.")
for key, bars in results.items():
    print(f"  {key[0]} {key[1]}: {len(bars)} bars")


## 7. Write to Catalog

Write all pulled bars to the ParquetDataCatalog. Also writes instrument definitions so BacktestNode can find them.


In [ ]:
catalog = ParquetDataCatalog(str(CATALOG_PATH))

# Write instrument definitions
for name, instrument in resolved_instruments.items():
    catalog.write_data([instrument])
    print(f"Wrote instrument: {instrument.id}")

# Write bar data
for (name, bar_spec), bars in results.items():
    if not bars:
        print(f"  {name} {bar_spec}: no bars to write — skipping")
        continue
    catalog.write_data(bars)
    print(f"  {name} {bar_spec}: wrote {len(bars)} bars")

print(f"\nAll data written to {CATALOG_PATH}")
print("Run the dashboard (bun run dev) to view instruments in the catalog.")


## 8. Verify

Check that the data is queryable from the catalog.


In [ ]:
from nautilus_trader.model.data import Bar

catalog = ParquetDataCatalog(str(CATALOG_PATH))

instruments = catalog.instruments()
print(f"Instruments in catalog: {len(instruments)}")
for inst in instruments:
    print(f"  {inst.id}")

bars = catalog.bars()
if bars:
    print(f"\nTotal bars: {len(bars)}")
    print(f"First: {bars[0].bar_type} @ {bars[0].ts_init}")
    print(f"Last:  {bars[-1].bar_type} @ {bars[-1].ts_init}")
else:
    print("\nNo bars in catalog")


## 9. Cleanup

Disconnect from TWS when done.


In [ ]:
client._client.disconnect()
print("Disconnected from TWS")
